# 02 - Data Cleaning & Feature Engineering

In [ ]:
from google.colab import drive
import os
import pandas as pd
import numpy as np

In [ ]:
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
FOLDER_PATH = "/content/drive/MyDrive/Raw_Dataset"


In [ ]:
# Get CSV files (sorted → consistent order)
csv_files = sorted([f for f in os.listdir(FOLDER_PATH) if f.endswith(".csv")])

assert len(csv_files) == 4, f"Expected 4 files, found {len(csv_files)}"

# Load into dictionary
dfs = {
    file.replace(".csv", ""): pd.read_csv(os.path.join(FOLDER_PATH, file))
    for file in csv_files
}

print("Loaded files:", list(dfs.keys()))

Loaded files: ['CA_youtube_trending_data', 'GB_youtube_trending_data', 'IN_youtube_trending_data', 'US_youtube_trending_data']


##1.Renaming columns

In [ ]:
for k, df in dfs.items():
    df.columns = df.columns.str.lower().str.strip()

    df.rename(columns={
        "publishedat": "publish_time",
        "channelid": "channel_id",
        "channeltitle": "channel_title",
        "categoryid": "category_id",
        "view_count": "views",
        "like_count": "likes",
        "comment_count": "comments",
    }, inplace=True)

    dfs[k] = df

In [ ]:
for k, df in dfs.items():
    display(df.head(2))

,video_id,title,publish_time,channel_id,channel_title,category_id,trending_date,tags,views,likes,dislikes,comments,thumbnail_link,comments_disabled,ratings_disabled,description
0,KX06ksuS6Xo,Diljit Dosanjh: CLASH (Official) Music Video |...,2020-08-11T07:30:02Z,UCZRdNleCgW-BGUJf-bbjzQg,Diljit Dosanjh,10,2020-08-12T00:00:00Z,clash diljit dosanjh|diljit dosanjh|diljit dos...,9140911,296541,6180,30059,https://i.ytimg.com/vi/KX06ksuS6Xo/default.jpg,False,False,CLASH official music video performed by DILJIT...
1,J78aPJ3VyNs,I left youtube for a month and THIS is what ha...,2020-08-11T16:34:06Z,UCYzPXprvl5Y-Sf0g4vX-m6g,jacksepticeye,24,2020-08-12T00:00:00Z,jacksepticeye|funny|funny meme|memes|jacksepti...,2038853,353797,2628,40222,https://i.ytimg.com/vi/J78aPJ3VyNs/default.jpg,False,False,I left youtube for a month and this is what ha...


,video_id,title,publish_time,channel_id,channel_title,category_id,trending_date,tags,views,likes,dislikes,comments,thumbnail_link,comments_disabled,ratings_disabled,description
0,J78aPJ3VyNs,I left youtube for a month and THIS is what ha...,2020-08-11T16:34:06Z,UCYzPXprvl5Y-Sf0g4vX-m6g,jacksepticeye,24,2020-08-12T00:00:00Z,jacksepticeye|funny|funny meme|memes|jacksepti...,2038853,353790,2628,40228,https://i.ytimg.com/vi/J78aPJ3VyNs/default.jpg,False,False,I left youtube for a month and this is what ha...
1,9nidKH8cM38,TAXI CAB SLAYER KILLS 'TO KNOW HOW IT FEELS',2020-08-11T20:00:45Z,UCFMbX7frWZfuWdjAML0babA,Eleanor Neale,27,2020-08-12T00:00:00Z,eleanor|neale|eleanor neale|eleanor neale true...,236830,16423,209,1642,https://i.ytimg.com/vi/9nidKH8cM38/default.jpg,False,False,The first 1000 people to click the link will g...


,video_id,title,publish_time,channel_id,channel_title,category_id,trending_date,tags,views,likes,dislikes,comments,thumbnail_link,comments_disabled,ratings_disabled,description
0,Iot0eF6EoNA,Sadak 2 | Official Trailer | Sanjay | Pooja | ...,2020-08-12T04:31:41Z,UCGqvJPRcv7aVFun-eTsatcA,FoxStarHindi,24,2020-08-12T00:00:00Z,sadak|sadak 2|mahesh bhatt|vishesh films|pooja...,9885899,224925,3979409,350210,https://i.ytimg.com/vi/Iot0eF6EoNA/default.jpg,False,False,Three Streams. Three Stories. One Journey. Sta...
1,x-KbnJ9fvJc,Kya Baat Aa : Karan Aujla (Official Video) Tan...,2020-08-11T09:00:11Z,UCm9SZAl03Rev9sFwloCdz1g,Rehaan Records,10,2020-08-12T00:00:00Z,[None],11308046,655450,33242,405146,https://i.ytimg.com/vi/x-KbnJ9fvJc/default.jpg,False,False,Singer/Lyrics: Karan Aujla Feat Tania Music/ D...


,video_id,title,publish_time,channel_id,channel_title,category_id,trending_date,tags,views,likes,dislikes,comments,thumbnail_link,comments_disabled,ratings_disabled,description
0,3C66w5Z0ixs,I ASKED HER TO BE MY GIRLFRIEND...,2020-08-11T19:20:14Z,UCvtRTOMP2TqYqu51xNrqAzg,Brawadis,22,2020-08-12T00:00:00Z,brawadis|prank|basketball|skits|ghost|funny vi...,1514614,156908,5855,35313,https://i.ytimg.com/vi/3C66w5Z0ixs/default.jpg,False,False,SUBSCRIBE to BRAWADIS ▶ http://bit.ly/Subscrib...
1,M9Pmf9AB4Mo,Apex Legends | Stories from the Outlands – “Th...,2020-08-11T17:00:10Z,UC0ZV6M2THA81QT9hrVWJG3A,Apex Legends,20,2020-08-12T00:00:00Z,Apex Legends|Apex Legends characters|new Apex ...,2381688,146739,2794,16549,https://i.ytimg.com/vi/M9Pmf9AB4Mo/default.jpg,False,False,"While running her own modding shop, Ramya Pare..."


## 2.Adding Country column

In [ ]:
country_map = {
    "CA_youtube_trending_data": "Canada",
    "GB_youtube_trending_data": "United Kingdom",
    "IN_youtube_trending_data": "India",
    "US_youtube_trending_data": "United States"
}

for k, df in dfs.items():
    df["country"] =  country_map[k]

In [ ]:
for k, df in dfs.items():
    display(df.head(2))

,video_id,title,publish_time,channel_id,channel_title,category_id,trending_date,tags,views,likes,dislikes,comments,thumbnail_link,comments_disabled,ratings_disabled,description,country
0,KX06ksuS6Xo,Diljit Dosanjh: CLASH (Official) Music Video |...,2020-08-11T07:30:02Z,UCZRdNleCgW-BGUJf-bbjzQg,Diljit Dosanjh,10,2020-08-12T00:00:00Z,clash diljit dosanjh|diljit dosanjh|diljit dos...,9140911,296541,6180,30059,https://i.ytimg.com/vi/KX06ksuS6Xo/default.jpg,False,False,CLASH official music video performed by DILJIT...,Canada
1,J78aPJ3VyNs,I left youtube for a month and THIS is what ha...,2020-08-11T16:34:06Z,UCYzPXprvl5Y-Sf0g4vX-m6g,jacksepticeye,24,2020-08-12T00:00:00Z,jacksepticeye|funny|funny meme|memes|jacksepti...,2038853,353797,2628,40222,https://i.ytimg.com/vi/J78aPJ3VyNs/default.jpg,False,False,I left youtube for a month and this is what ha...,Canada


,video_id,title,publish_time,channel_id,channel_title,category_id,trending_date,tags,views,likes,dislikes,comments,thumbnail_link,comments_disabled,ratings_disabled,description,country
0,J78aPJ3VyNs,I left youtube for a month and THIS is what ha...,2020-08-11T16:34:06Z,UCYzPXprvl5Y-Sf0g4vX-m6g,jacksepticeye,24,2020-08-12T00:00:00Z,jacksepticeye|funny|funny meme|memes|jacksepti...,2038853,353790,2628,40228,https://i.ytimg.com/vi/J78aPJ3VyNs/default.jpg,False,False,I left youtube for a month and this is what ha...,United Kingdom
1,9nidKH8cM38,TAXI CAB SLAYER KILLS 'TO KNOW HOW IT FEELS',2020-08-11T20:00:45Z,UCFMbX7frWZfuWdjAML0babA,Eleanor Neale,27,2020-08-12T00:00:00Z,eleanor|neale|eleanor neale|eleanor neale true...,236830,16423,209,1642,https://i.ytimg.com/vi/9nidKH8cM38/default.jpg,False,False,The first 1000 people to click the link will g...,United Kingdom


,video_id,title,publish_time,channel_id,channel_title,category_id,trending_date,tags,views,likes,dislikes,comments,thumbnail_link,comments_disabled,ratings_disabled,description,country
0,Iot0eF6EoNA,Sadak 2 | Official Trailer | Sanjay | Pooja | ...,2020-08-12T04:31:41Z,UCGqvJPRcv7aVFun-eTsatcA,FoxStarHindi,24,2020-08-12T00:00:00Z,sadak|sadak 2|mahesh bhatt|vishesh films|pooja...,9885899,224925,3979409,350210,https://i.ytimg.com/vi/Iot0eF6EoNA/default.jpg,False,False,Three Streams. Three Stories. One Journey. Sta...,India
1,x-KbnJ9fvJc,Kya Baat Aa : Karan Aujla (Official Video) Tan...,2020-08-11T09:00:11Z,UCm9SZAl03Rev9sFwloCdz1g,Rehaan Records,10,2020-08-12T00:00:00Z,[None],11308046,655450,33242,405146,https://i.ytimg.com/vi/x-KbnJ9fvJc/default.jpg,False,False,Singer/Lyrics: Karan Aujla Feat Tania Music/ D...,India


,video_id,title,publish_time,channel_id,channel_title,category_id,trending_date,tags,views,likes,dislikes,comments,thumbnail_link,comments_disabled,ratings_disabled,description,country
0,3C66w5Z0ixs,I ASKED HER TO BE MY GIRLFRIEND...,2020-08-11T19:20:14Z,UCvtRTOMP2TqYqu51xNrqAzg,Brawadis,22,2020-08-12T00:00:00Z,brawadis|prank|basketball|skits|ghost|funny vi...,1514614,156908,5855,35313,https://i.ytimg.com/vi/3C66w5Z0ixs/default.jpg,False,False,SUBSCRIBE to BRAWADIS ▶ http://bit.ly/Subscrib...,United States
1,M9Pmf9AB4Mo,Apex Legends | Stories from the Outlands – “Th...,2020-08-11T17:00:10Z,UC0ZV6M2THA81QT9hrVWJG3A,Apex Legends,20,2020-08-12T00:00:00Z,Apex Legends|Apex Legends characters|new Apex ...,2381688,146739,2794,16549,https://i.ytimg.com/vi/M9Pmf9AB4Mo/default.jpg,False,False,"While running her own modding shop, Ramya Pare...",United States


##3.Merging datasets

In [ ]:
df = pd.concat(dfs.values(), ignore_index=True)

print("Final Shape:", df.shape)

Final Shape: (1057597, 17)


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1057597 entries, 0 to 1057596
Data columns (total 17 columns):
 #   Column             Non-Null Count    Dtype 
---  ------             --------------    ----- 
 0   video_id           1057597 non-null  object
 1   title              1057597 non-null  object
 2   publish_time       1057597 non-null  object
 3   channel_id         1057597 non-null  object
 4   channel_title      1057596 non-null  object
 5   category_id        1057597 non-null  int64 
 6   trending_date      1057597 non-null  object
 7   tags               1057597 non-null  object
 8   views              1057597 non-null  int64 
 9   likes              1057597 non-null  int64 
 10  dislikes           1057597 non-null  int64 
 11  comments           1057597 non-null  int64 
 12  thumbnail_link     1057597 non-null  object
 13  comments_disabled  1057597 non-null  bool  
 14  ratings_disabled   1057597 non-null  bool  
 15  description        1024762 non-null  object
 16  

In [ ]:
df.describe()

,category_id,views,likes,dislikes,comments
count,1.057597e+06,1.057597e+06,1.057597e+06,1.057597e+06,1.057597e+06
mean,1.944138e+01,2.525750e+06,1.243993e+05,1.387194e+03,8.800175e+03
std,6.518520e+00,7.596333e+06,3.906731e+05,3.576234e+04,6.717003e+04
min,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.700000e+01,4.118350e+05,1.442700e+04,0.000000e+00,8.040000e+02
50%,2.200000e+01,8.864000e+05,3.754500e+04,0.000000e+00,2.102000e+03
75%,2.400000e+01,2.087803e+06,9.965400e+04,4.400000e+02,5.419000e+03
max,2.900000e+01,1.407644e+09,1.611524e+07,1.234147e+07,6.738565e+06


##4.Changing datatypes

In [ ]:
df["publish_time"] = pd.to_datetime(df["publish_time"], errors="coerce")
df["trending_date"] = pd.to_datetime(df["trending_date"], errors="coerce")

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1057597 entries, 0 to 1057596
Data columns (total 17 columns):
 #   Column             Non-Null Count    Dtype              
---  ------             --------------    -----              
 0   video_id           1057597 non-null  object             
 1   title              1057597 non-null  object             
 2   publish_time       1057597 non-null  datetime64[ns, UTC]
 3   channel_id         1057597 non-null  object             
 4   channel_title      1057596 non-null  object             
 5   category_id        1057597 non-null  int64              
 6   trending_date      1057597 non-null  datetime64[ns, UTC]
 7   tags               1057597 non-null  object             
 8   views              1057597 non-null  int64              
 9   likes              1057597 non-null  int64              
 10  dislikes           1057597 non-null  int64              
 11  comments           1057597 non-null  int64              
 12  thumbnail_link

In [ ]:
df[["publish_time", "trending_date"]].isnull().sum()

,0
publish_time,0
trending_date,0


In [ ]:
df["channel_title"].isnull().sum()

np.int64(1)

##5.Handling Missing Null values

In [ ]:
df["description"].isnull().sum()

np.int64(32835)

In [ ]:
df["tags"].value_counts()

,count
tags,
[None],175748
sidemen|moresidemen|miniminter|ksi|zerkaa|behzinga|tbjzl|vikkstar|wroetoshaw,1803
Amber|amber vtuber|genshi|genshi game|genshi impact|genshi video|genshin|genshin game|genshin impact|genshin impact 2020|genshin impact game|genshin impact good|genshin impact graphics|genshin impact introduction|genshin impact manga|genshin impact wiki|geshin|geshin game|Teyvat|yuanshen game|miHoYo China|miHoYo Japan|adventure story|open world game|anime style|MMORPG|anime games|mobile game|MMO PlayStation|yt:cc=on,1540
sidemen|sidemen sunday|#sidemensunday,1527
HYBE|HYBE LABELS|하이브|하이브레이블즈,1491
...,...
technician form online|technician form fill up|technician form fill up 2024|technician form 2024|technician form kaise bhare|technician form kaise bharen|technician form|rrb technician form fill up 2024|rrb technician form fill up|rrb technician form fill up online|rrb technician|rrb technician vacancy 2024|technician form fill|technician|technician zone wise form fill up|railway technician form fill up|rrb technician online form 2024|rrb technician form|mdcl|🔥,1
haryanvi songs|new haryanvi songs|latest haryanvi songs|haryanvi songs haryanavi|new haryanvi songs haryanavi 2023|new haryanvi song 2023|latest haryanvi songs 2023|ndee kundu new song|notorious desi|ndee kundu haryanvi song|new haryanvi song latest this week|deshi song|ndee kundu haryanvi songs|mat samjhe shareef inn desiyan ne|ndee kundu latest song|naam tera|leke mere kale kale car darling|ndee kundu notorious desi|new haryanvi,1
shivam dikro new video|shivam dikro car video|shivam dikro racing video|shivam dikro dada ji|viral|dikro ke dada ji|dada ji ki kahaniya|dahej pratha|shaadi ki car|shivam dikro race challenge|car race|thar offroading|mahindra new thar|thar comedy video|thar ki chori|thar race|thar thug of war|shivam dikro thar|dikro ki that|black thar modified|shivam dikro|Dada Ji Ki Thar Car || Thar Ki Chori || Shivam Dikro|thar part 2|dadaji part 2|new video dikro,1


In [ ]:
df["channel_title"].fillna("Unknown", inplace=True)
df["description"].fillna("No Description", inplace=True)
df["tags"] = df["tags"].replace("[None]", "")

/tmp/ipykernel_10633/4266697393.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["channel_title"].fillna("Unknown", inplace=True)
/tmp/ipykernel_10633/4266697393.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)',

##6.Removing duplicates

In [ ]:
#YouTube removed dislikes
df = df.drop(columns=["dislikes"])

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1057597 entries, 0 to 1057596
Data columns (total 16 columns):
 #   Column             Non-Null Count    Dtype              
---  ------             --------------    -----              
 0   video_id           1057597 non-null  object             
 1   title              1057597 non-null  object             
 2   publish_time       1057597 non-null  datetime64[ns, UTC]
 3   channel_id         1057597 non-null  object             
 4   channel_title      1057597 non-null  object             
 5   category_id        1057597 non-null  int64              
 6   trending_date      1057597 non-null  datetime64[ns, UTC]
 7   tags               1057597 non-null  object             
 8   views              1057597 non-null  int64              
 9   likes              1057597 non-null  int64              
 10  comments           1057597 non-null  int64              
 11  thumbnail_link     1057597 non-null  object             
 12  comments_disab

In [ ]:
df.duplicated().sum()

np.int64(395)

In [ ]:
df = df.drop_duplicates()

##7.Feature Engineering

In [ ]:
# Engagement Rate (influence)
df["engagement_rate"] = (df["likes"] + df["comments"]) / df["views"]


# Time to trend (days)
df["time_to_trend"] = (
    df["trending_date"] - df["publish_time"]
).dt.days

# Publish hour & day
df["publish_hour"] = df["publish_time"].dt.hour
df["publish_day"] = df["publish_time"].dt.day_name()

# Weekend indicator
df["is_weekend"] = df["publish_day"].isin(["Saturday", "Sunday"]).astype(int)



# Title length
df["title_length"] = df["title"].apply(len)

# Uppercase ratio in title
df["title_uppercase_ratio"] = df["title"].apply(
    lambda x: sum(1 for c in x if c.isupper()) / len(x) if len(x) > 0 else 0
)

# Exclamation presence
df["title_has_exclamation"] = df["title"].str.contains("!").astype(int)



# Tag count
df["tag_count"] = df["tags"].apply(
    lambda x: 0 if x == "" else x.count("|") + 1
)

# Has tags or not
df["has_tags"] = (df["tag_count"] > 0).astype(int)



# Virality score (views per time)
df["virality_score"] = df["views"] / (df["time_to_trend"] + 1)

# Like ratio
df["like_ratio"] = df["likes"] / df["views"]

# Comment ratio
df["comment_ratio"] = df["comments"] / df["views"]

In [ ]:
df[df["views"] < 0].shape

(0, 29)

In [ ]:
 df[df["time_to_trend"] <0].shape

(14818, 29)

In [ ]:
 df[df["time_to_trend"] ==np.inf].shape

(0, 29)

In [ ]:
df = df[df["time_to_trend"] >= 0]

In [ ]:
# Replacing infinite values
df.replace([np.inf, -np.inf], 0, inplace=True)


In [ ]:
df[[
    "engagement_rate",
    "virality_score",
    "like_ratio",
    "comment_ratio",
    "time_to_trend"
]].describe()

,engagement_rate,virality_score,like_ratio,comment_ratio,time_to_trend
count,1.042227e+06,1.042384e+06,1.042200e+06,1.042218e+06,1.042384e+06
mean,5.526048e-02,7.599346e+05,5.158949e-02,3.672360e-03,2.753616e+00
std,3.928474e-02,2.224964e+06,3.690080e-02,5.099952e-03,2.312514e+00
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,2.564722e-02,1.320043e+05,2.347384e-02,1.198788e-03,1.000000e+00
50%,4.835095e-02,2.872694e+05,4.522772e-02,2.513347e-03,2.000000e+00
75%,7.430828e-02,6.908400e+05,6.988578e-02,4.574689e-03,4.000000e+00
max,7.317551e-01,7.038218e+08,4.575602e-01,4.266585e-01,3.600000e+01


##Saving as csv

In [ ]:

df.to_csv("/content/drive/MyDrive/DVA_Project/data/processed/cleaned_dataset.csv", index=False)